In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
import pandas as pd
import matplotlib.pyplot as plt
import random
from PIL import Image
from datetime import datetime
import os
from pathlib import Path

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

import models
import utils
import control_utils
from live_simulation_pyqt import get_rest_state_from_steady
from importlib import reload
from IPython.display import clear_output
reload(models)
reload(control_utils)

print("Using device:", device)

torch.set_default_dtype(torch.float32)

plt.rcParams.update({
    'text.usetex': True,
    'font.family': 'serif',
    'font.serif': ['Times New Roman'],
    'axes.labelsize': 8,
    'xtick.labelsize': 7,
    'ytick.labelsize': 7,
    'legend.fontsize': 7,
    'font.size': 8
})

# Load models and configs

In [ ]:

# Model names to search for
model_config_names = {
    "1 Seg Koopman": "scr_match_1seg_koopman",
    "1 Seg Koopman + Attention": "scr_match_1seg_koopman_attn",
    "1 Seg MLP": "scr_match_1seg_mlp",
    "1 Seg MLP + Attention": "scr_match_1seg_mlp_attn",
    "1 Seg Oscillator": "scr_match_1seg_harmonic1d",
    "1 Seg Oscillator + Attention": "scr_match_1seg_harmonic2d_vel_attn",

    "2 Seg Koopman": "scr_match_2seg_koopman",
    "2 Seg Koopman + Attention": "scr_match_2seg_koopman_attn",
    "2 Seg MLP": "scr_match_2seg_mlp",
    "2 Seg MLP + Attention": "scr_match_2seg_mlp_attn",
    "2 Seg Oscillator": "scr_match_2seg_harmonic1d_main",
    "2 Seg Oscillator + Attention": "scr_match_2seg_harmonic2d_vel_attn"
}

model_plot_names_separate = {
    "1 Seg MLP": "MLP",
    "1 Seg MLP + Attention": "MLP + ABCD",
    "1 Seg Koopman": "Koopman",
    "1 Seg Koopman + Attention": "Koopman + ABCD",
    "1 Seg Oscillator": "Oscillators",
    "1 Seg Oscillator + Attention": "2D Oscillators + ABCD",
    "2 Seg Koopman": "Koopman",
    "2 Seg Koopman + Attention": "Koopman + ABCD",
    "2 Seg Oscillator": "Oscillators",
    "2 Seg Oscillator + Attention": "2D Oscillators + ABCD"
}

# Find trained model directories
results_dir = Path("results/models/V02_CDC/")

def find_latest_model_run(config_name):
    # Only match directories whose name ends with the pattern after a variable date (e.g. "20240606T1048_scr_match_1seg_koopman")
    # To avoid substring issues, require exact end-match.
    matching_dirs = sorted([
        d for d in results_dir.iterdir()
        if d.is_dir() and d.name.endswith(config_name)
    ])
    return matching_dirs[-1] if matching_dirs else None

model_dirs = {}
for name, config_name in model_config_names.items():
    model_dir = find_latest_model_run(config_name)
    if model_dir:
        model_dirs[name] = model_dir
        print(f"✓ {name}: {model_dir.name}")
    else:
        print(f"✗ {name} not found")

# Load configs from epoch folders
import json

configs = {}
for name in model_dirs.keys():
    epoch_folders = sorted([d for d in model_dirs[name].glob("epoch_*") if d.is_dir()])
    if epoch_folders:
        config_file = epoch_folders[-1] / "config.json"
        if config_file.exists():
            with open(config_file, 'r') as f:
                configs[name] = json.load(f)
            print(f"✓ Loaded config for {name}")
        else:
            print(f"✗ Config not found for {name}")

In [ ]:
# Config analysis

# INSERT_YOUR_CODE

from collections import defaultdict
import pprint

# Collect all keys across all configs
all_keys = set()
for cfg in configs.values():
    all_keys.update(cfg.keys())

# Find keys where values differ between configs
diff_keys = []
for key in sorted(all_keys):
    values = []
    for name in configs.keys():
        val = configs[name].get(key, None)
        values.append(val)
    # If not all equal, consider this a diff key
    if len(set([str(v) for v in values])) > 1:
        diff_keys.append(key)

if diff_keys:
    print("Config keys with differing values among models:")
    table = []
    header = ["Model"] + diff_keys
    table.append(header)
    for name in configs.keys():
        row = [name]
        for key in diff_keys:
            val = configs[name].get(key, None)
            row.append(repr(val))
        table.append(row)
    # Print as a nice table
    col_widths = [max(len(str(cell)) for cell in col) for col in zip(*table)]
    for i, row in enumerate(table):
        print(" | ".join(str(cell).ljust(width) for cell, width in zip(row, col_widths)))
        if i == 0:
            print("-+-".join('-'*width for width in col_widths))
else:
    print("No differing config values found among models.")


In [ ]:
# Initialize models
vaes = {}
dynamics_models = {}

for name, config in configs.items():
    # Create VAE
    vae = models.VAE(
        input_channels=1,  # Grayscale images
        latent_dim=config["latent_dim"],
        is_vae=config["use_vae"],
        use_attention_decoder=config["use_attention_decoder"],
        attention_feature_dim=config["attention_feature_dim"],
        attention_downsample_factor=config["attention_downsample_factor"],
        background_token_value=config["background_token_value"],
        image_size=config["resolution"],
        dynamics_type=config["dynamics_type"],
        gumbel_noise_strength=config["gumbel_noise_strength"]
    ).to(device)
    
    # Create dynamics model
    dynamics_config = config.copy()
    num_delays = config.get("num_actuation_delays", 1)
    if num_delays > 1:
        dynamics_config["actuation_dim"] = config["actuation_dim"] * num_delays
    
    latent_dynamics = models.create_dynamics_model(
        config=dynamics_config,
        device=device
    ).to(device)
    
    vaes[name] = vae
    dynamics_models[name] = latent_dynamics
    print(f"✓ Initialized {name}")


In [ ]:
# Load trained checkpoints
for name in model_dirs.keys():
    # Find latest epoch folder
    epoch_folders = sorted([d for d in model_dirs[name].glob("epoch_*") if d.is_dir()])
    if epoch_folders:
        checkpoint_path = epoch_folders[-1] / "model_checkpoint.pt"
        if checkpoint_path.exists():
            checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=True)
            vaes[name].load_state_dict(checkpoint['vae_state_dict'])
            dynamics_models[name].load_state_dict(checkpoint['latent_dynamics_state_dict'])
            print(f"✓ Loaded {name} from epoch {checkpoint.get('epoch', '?')}")
        else:
            print(f"✗ Checkpoint not found for {name}")
    else:
        print(f"✗ No epoch folders for {name}")

# Set models to eval mode
for vae in vaes.values():
    vae.eval()
for dyn in dynamics_models.values():
    dyn.eval()


In [ ]:
# Load all the target states manually created on the simulator

simulator_pickle_paths_1seg = [
    "results/control_states/1seg/1Seg_Koopman_Static_Extrapolated_1.pkl",
    "results/control_states/1seg/1Seg_Koopman_Dynamic_Normal_1.pkl",
    "results/control_states/1seg/1Seg_Koopman_Dynamic_Normal_2.pkl",
    "results/control_states/1seg/1Seg_Koopman_Dynamic_Normal_3.pkl",
    "results/control_states/1seg/1Seg_Koopman_Dynamic_Normal_4.pkl",
    "results/control_states/1seg/1Seg_Koopman_Dynamic_Normal_5.pkl",
    "results/control_states/1seg/1Seg_Koopman_Dynamic_HighDyn_1.pkl",
    "results/control_states/1seg/1Seg_Koopman_Dynamic_HighDyn_2.pkl",
    "results/control_states/1seg/1Seg_Koopman_Dynamic_HighDyn_3.pkl",
    "results/control_states/1seg/1Seg_Koopman_Dynamic_Upswing_1.pkl",
    "results/control_states/1seg/1Seg_Koopman_Dynamic_Upswing_2.pkl",
    "results/control_states/1seg/1Seg_Koopman_Dynamic_Upswing_3.pkl",    
    ]

simulator_pickle_paths_2seg = [
    "results/control_states/2seg/2Seg_Koopman_Static_Extrapolated_1.pkl",
    "results/control_states/2seg/2Seg_Koopman_Dynamic_Normal_1.pkl",
    "results/control_states/2seg/2Seg_Koopman_Dynamic_Normal_2.pkl",
    "results/control_states/2seg/2Seg_Koopman_Dynamic_Normal_3.pkl",
    "results/control_states/2seg/2Seg_Koopman_Dynamic_Normal_4.pkl",
    "results/control_states/2seg/2Seg_Koopman_Dynamic_Normal_5.pkl",
    "results/control_states/2seg/2Seg_Koopman_Dynamic_HighDyn_1.pkl",
    "results/control_states/2seg/2Seg_Koopman_Dynamic_HighDyn_2.pkl",
    "results/control_states/2seg/2Seg_Koopman_Dynamic_HighDyn_3.pkl",
    "results/control_states/2seg/2Seg_Koopman_Dynamic_Upswing_1.pkl",
    "results/control_states/2seg/2Seg_Koopman_Dynamic_Upswing_2.pkl",
    "results/control_states/2seg/2Seg_Koopman_Dynamic_Upswing_3.pkl",    
    "results/control_states/2seg/2Seg_2DOscillator_Dynamic_Long_1.pkl",
    "results/control_states/2seg/2Seg_2DOscillator_Dynamic_Long_2.pkl",
    "results/control_states/2seg/2Seg_2DOscillator_Dynamic_Long_3.pkl",
    "results/control_states/2seg/2Seg_2DOscillator_Dynamic_Long_4.pkl",
    "results/control_states/2seg/2Seg_2DOscillator_Dynamic_Long_5.pkl",    
    ]

vaes_1seg = {k: v for k, v in vaes.items() if "1 Seg" in k}
configs_1seg = {k: v for k, v in configs.items() if "1 Seg" in k}
simulator_states_1seg = control_utils.load_simulator_pickles_and_compute_latents(simulator_pickle_paths_1seg, vaes_1seg, configs_1seg, device)

vaes_2seg = {k: v for k, v in vaes.items() if "2 Seg" in k}
configs_2seg = {k: v for k, v in configs.items() if "2 Seg" in k}
simulator_states_2seg = control_utils.load_simulator_pickles_and_compute_latents(simulator_pickle_paths_2seg, vaes_2seg, configs_2seg, device)
#Access: simulator_states['Koopman_Dynamic_Upswing_1']['2 Seg Oscillator + Attention']['z']  # (n_states, latent_dim)

# Load dataset

In [ ]:
# Load low-res (32x32) and high-res (128x128) datasets for 1-seg and 2-seg
resolutions = [32]
datasets_static = {
    res: {
        "scr_match_1seg": f"1segment_step_input_rand_compressed_static_processed.npz",
        "scr_match_2seg": f"2segments_step_input_rand_18s_compressed_static_processed.npz",
    } for res in resolutions
}
datasets_dyn = {
    res: {
        "scr_match_1seg": f"1segment_smooth_input_rand_compressed_processed.npz",
        "scr_match_2seg": f"2segments_smooth_input_rand_compressed_processed.npz",
    } for res in resolutions
}

# Load low-res datasets (full)
dataset_1seg_static = np.load(f"data/scr_match2/{datasets_static[resolutions[0]]['scr_match_1seg']}")
#dataset_1seg_dyn = np.load(f"data/scr_match2/{datasets_dyn[resolutions[0]]['scr_match_1seg']}")
dataset_2seg_static = np.load(f"data/scr_match2/{datasets_static[resolutions[0]]['scr_match_2seg']}")
#dataset_2seg_dyn = np.load(f"data/scr_match2/{datasets_dyn[resolutions[0]]['scr_match_2seg']}")


#dataset_2seg = np.load(f"data/{datasets[resolutions[0]]['scr_match_2seg']}")

# Load high-res datasets (validation-only)
# dataset_1seg_highres = np.load(f"data/{datasets[resolutions[1]]['scr_match_1seg'].replace('.npz', '_val.npz')}")
# dataset_2seg_highres = np.load(f"data/{datasets[resolutions[1]]['scr_match_2seg'].replace('.npz', '_val.npz')}")

print("✓ Loaded low-res datasets (full)")
print(f"  1-seg: {dataset_1seg_static['images'].shape}")

print(f"  2-seg: {dataset_2seg_static['images'].shape}")
#print(f"  2-seg: {dataset_2seg_dyn['images'].shape}")
# print("✓ Loaded high-res datasets (validation-only)")
# print(f"  1-seg: {dataset_1seg_highres['images'].shape}")
# print(f"  2-seg: {dataset_2seg_highres['images'].shape}")

In [ ]:
# Prepare validation data for both 1-seg and 2-seg datasets

# Separate configs by dataset type
configs_1seg = {name: cfg for name, cfg in configs.items() if "1seg" in cfg["dataset"]}
configs_2seg = {name: cfg for name, cfg in configs.items() if "2seg" in cfg["dataset"]}

if configs_1seg:
    all_o_1seg_static, all_u_1seg_static = control_utils.prepare_static_data(dataset_1seg_static, configs_1seg)
#     all_o_1seg_dyn, all_u_1seg_dyn = control_utils.prepare_dynamic_val_data(dataset_1seg_dyn, configs_1seg)

if configs_2seg:
    all_o_2seg_static, all_u_2seg_static = control_utils.prepare_static_data(dataset_2seg_static, configs_2seg)
    # all_o_2seg_dyn, all_u_2seg_dyn = control_utils.prepare_dynamic_val_data(dataset_2seg_dyn, configs_2seg)

# Prepare validation data for 2-segment dataset  
# print("\n=== 2-Segment Dataset ===")
# val_o_2seg, val_u_2seg = prepare_val_data(dataset_2seg, configs_2seg)
# print(f"✓ Validation set: {val_o_2seg.shape[0]} samples")
# print(f"✓ Actuation shape: {val_u_2seg.shape}")
# print(f"✓ Models: {list(configs_2seg.keys())}")

# Calculate latent states and velocities

In [ ]:
# Precompute latent states and velocities for full validation sets
# (processed in batches, but covers the entire validation split)
from typing import Dict, Tuple

def compute_static_latents_for_validation(
    val_o: torch.Tensor,
    model_name: str,
    batch_size: int = 256,
) -> torch.Tensor:
    """Return z for every frame in val_o for the given model (static: single-frame)."""
    vae = vaes[model_name]
    config = configs[model_name]

    n_samples = val_o.shape[0]
    latent_dim = config["latent_dim"]

    # Preallocate on CPU to avoid GPU memory spikes
    z_all = torch.empty(n_samples, latent_dim, dtype=torch.float32)

    vae.eval()
    with torch.no_grad():
        for start in range(0, n_samples, batch_size):
            end = min(start + batch_size, n_samples)

            o_batch = val_o[start:end].to(device)

            if config["use_vae"]:
                _, _, mu_batch, _ = vae(o_batch)
            else:
                _, mu_batch = vae(o_batch)

            z_all[start:end] = mu_batch.detach().cpu()

    return z_all

def compute_dynamic_latents_for_validation(
    val_o: torch.Tensor,
    model_name: str,
    delta_t: float = None,
    batch_size: int = 256,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Compute (z, z_dot) for every frame in val_o for the given model.
    z: latent state (one per frame)
    z_dot: latent velocity (one per frame; velocity is calculated using latent_velocity_from_observation_velocity)
    """
    vae = vaes[model_name]
    config = configs[model_name]
    n_samples = val_o.shape[0]
    latent_dim = config["latent_dim"]

    # allow explicit delta_t passed in, or get from config.
    if delta_t is None:
        delta_t = config.get("delta_t", 1.0)

    # Get all latents for all frames (mu values)
    z_all = torch.empty(n_samples, latent_dim, dtype=torch.float32)
    vae.eval()
    with torch.no_grad():
        for start in range(0, n_samples, batch_size):
            end = min(start + batch_size, n_samples)
            o_batch = val_o[start:end].to(device)
            if config["use_vae"]:
                _, _, mu_batch, _ = vae(o_batch)
            else:
                _, mu_batch = vae(o_batch)
            z_all[start:end] = mu_batch.detach().cpu()
    
    # Prepare z_dot_all, same size as z_all
    z_dot_all = torch.zeros(n_samples, latent_dim, dtype=torch.float32)
    # Calculate o_dot using central differences (except at boundaries: forward/backward)
    # Then use vae.latent_velocity_from_observation_velocity for each middle frame
    val_o_cpu = val_o.cpu()
    with torch.no_grad():
        # For middle points (central difference)
        for i in range(1, n_samples - 1):
            o_prev = val_o_cpu[i - 1 : i]
            o_next = val_o_cpu[i + 1 : i + 2]
            o_curr = val_o_cpu[i : i + 1]
            o_dot = (o_next - o_prev) / (2.0 * delta_t)
            # Compute latent velocity with the VAE's method (Jacobian-based)
            z_dot = vae.latent_velocity_from_observation_velocity(o_curr.to(device), o_dot.to(device))
            z_dot_all[i] = z_dot.detach().cpu().squeeze(0)
        # For first point (forward difference)
        if n_samples > 1:
            o_dot0 = (val_o_cpu[1:2] - val_o_cpu[0:1]) / delta_t
            z_dot0 = vae.latent_velocity_from_observation_velocity(val_o_cpu[0:1].to(device), o_dot0.to(device))
            z_dot_all[0] = z_dot0.detach().cpu().squeeze(0)
        # For last point (backward difference)
        if n_samples > 1:
            o_dotN = (val_o_cpu[-1:] - val_o_cpu[-2:-1]) / delta_t
            z_dotN = vae.latent_velocity_from_observation_velocity(val_o_cpu[-1:].to(device), o_dotN.to(device))
            z_dot_all[-1] = z_dotN.detach().cpu().squeeze(0)
        # If n_samples == 1: leave z_dot_all as zeros 

    return z_all, z_dot_all

latent_states_static_normal: Dict[str, torch.Tensor] = {}
latent_states_dyn: Dict[str, Tuple[torch.Tensor, torch.Tensor]] = {}

# Compute for all models on their matching validation set
for model_name in vaes.keys():
    if "1seg" in configs[model_name]["dataset"]:
        all_o = all_o_1seg_static
        all_u = all_u_1seg_static
    elif "2seg" in configs[model_name]["dataset"]:
        all_o = all_o_2seg_static
        all_u = all_u_2seg_static
        #all_o_dyn = all_o_2seg_dyn
    else:
        print(f"⚠ Unknown dataset for {model_name}")
        continue

    print(f"Computing (static) latents for {model_name} on {all_o.shape[0]} frames...")
    z_all_static = compute_static_latents_for_validation(all_o, model_name)
    latent_states_static_normal[model_name] = z_all_static
    print(f"✓ Done: z {z_all_static.shape}")

    # Compute dynamic latents if dynamic validation data available
    # if all_o_dyn is not None:
    #     print(f"Computing (dynamic) latents and velocities for {model_name} on {all_o_dyn.shape[0]} frames...")
    #     z_dyn, z_dot_dyn = compute_dynamic_latents_for_validation(all_o_dyn, model_name)
    #     latent_states_dyn[model_name] = (z_dyn, z_dot_dyn)
    #     print(f"✓ Done: z {z_dyn.shape}, z_dot {z_dot_dyn.shape}")

print("✓ Precomputed latents for all validation data")


In [ ]:

z_stats = {}
for model_name, z in latent_states_static_normal.items():
    z_mean = z.mean()
    z_std = z.std()
    z_stats[model_name] = {"z_mean": z_mean, "z_std": z_std}
    
z_stats

# Finalize full target dict

In [ ]:
all_target_states = simulator_states_2seg.copy()
all_target_states.update(simulator_states_1seg)

all_target_states['Static_Normal'] = {}
all_target_states['Static_Normal']['z'] = latent_states_static_normal

# For 'u' and 'o', create per-model dicts with the correct data matching model type
all_target_states['Static_Normal']['u'] = {}
all_target_states['Static_Normal']['o'] = {}
for model_name in latent_states_static_normal.keys():
    if "1seg" in configs[model_name]["dataset"]:
        all_target_states['Static_Normal']['u'][model_name] = all_u_1seg_static
        all_target_states['Static_Normal']['o'][model_name] = all_o_1seg_static
    elif "2seg" in configs[model_name]["dataset"]:
        all_target_states['Static_Normal']['u'][model_name] = all_u_2seg_static
        all_target_states['Static_Normal']['o'][model_name] = all_o_2seg_static
    else:
        print(f"⚠ Warning: Unknown dataset for model {model_name}, not assigning 'u' or 'o'.")

# Rest states per model (same as simulator: first steady-state frame from dataset, encoded; z_dot = 0)
all_target_states['Rest_States'] = {}
for model_name in latent_states_static_normal.keys():
    z_rest, z_dot_rest = get_rest_state_from_steady(vaes[model_name], configs[model_name], device)
    all_target_states['Rest_States'][model_name] = {'z': z_rest, 'z_dot': z_dot_rest}



# Control tests

In [ ]:
model_names = list(model_config_names.keys())

## Static Normal + Extrapolated

### Single

In [ ]:
for j in range(11,12):
    model_name = model_names[j]  # or pick one
    print(model_name)
    dyn = dynamics_models[model_name]
    dev = next(dyn.parameters()).device
    dt = configs[model_name]["delta_t"]


    if "1 Seg" in model_name:
        T = 100
        n_iter = 300
        lr = 1e-1
        scheduler_step_size = 500
        scheduler_gamma = 0.1

        w_Q=1.0
        w_dQ=.1*dt
        w_Qf=.0
        w_dQf=.0*dt
        w_R=1e2
        delta_u_max=1.0*0.02
        w_du_bound=1e3

        dim_u=2
        dim_z=8

    elif "2 Seg" in model_name:
        T = 100
        n_iter = 300
        lr = 1e-1
        scheduler_step_size = 500
        scheduler_gamma = 0.1

        w_Q=1.0
        w_dQ=.1*dt
        w_Qf=.0
        w_dQf=.0*dt
        w_R=1e0
        delta_u_max=1.0*0.02
        w_du_bound=1e2

        dim_u=4
        dim_z=10


    z_rest = all_target_states['Rest_States'][model_name]['z'].squeeze(0).to(dev)
    u_rest=torch.ones(dim_u,device=dev)*0.01

    current_target_state=2

    target_file = 'Static_Normal'

    if "Normal" in target_file:
        type = 'normal'
    elif "Extrapolated" in target_file:
        type = 'extrapolated'

    if type == 'normal':
        all_zs=all_target_states[target_file]['z'][model_name][:8]
        all_us=all_target_states[target_file]['u'][model_name][:8]
        u_final=all_us[current_target_state].to(dev)

    elif type == 'extrapolated':
        all_zs=all_target_states[target_file][model_name]['z']
        all_us=None
        u_final=None

    # Use same z_dim/to_latent as multi-step so step-0 target matches exactly
    def to_latent_static(z):
        z = z.to(dev) if hasattr(z, 'to') else torch.tensor(z, device=dev)
        if z.shape[-1] > dim_z:
            return z[..., :dim_z].clone()
        return z

    z0=z_rest
    u_init=u_rest


    zd0 = torch.zeros(dim_z,device=dev) # Should always start at 0 speed

    oT = all_target_states['Static_Normal']['o'][model_name][current_target_state].to(dev)
    zT = to_latent_static(all_zs[current_target_state])
    zdT = torch.zeros_like(zT,device=dev)

    result_static = control_utils.optimize_open_loop_control(
        dyn=dyn,
        z0=z0,
        zd0=zd0,
        zT=zT,
        zdT=zdT,
        u_init=u_init,
        T=T,
        dt=dt,
        n_iter=n_iter,
        lr=lr,
        scheduler_step_size=scheduler_step_size,
        scheduler_gamma=scheduler_gamma,
        fix_u0_to_uref=True,
        use_sigmoid=True,
        z_ref_traj=None,
        zd_ref_traj=None,
        w_Q=w_Q,
        w_dQ=w_dQ,
        w_Qf=w_Qf,
        w_dQf=w_dQf,
        w_R=w_R,
        device=dev,
        z_stats=z_stats.get(model_name) if "z_stats" in dir() else None,
        delta_u_max=delta_u_max,
        w_du_bound=w_du_bound,
        u_max=0.9,
    )

    u_seq_static = result_static["u_seq"]
    print(f"Static control: final terminal error = {torch.norm(result_static['zT_pred'] - zT).item():.4f}")

    # saved_path = control_utils.save_control_output(
#     model_name, target_file, result_static,
#     simulator_pickle_paths=simulator_pickle_paths_2seg,
# )
# print(f"Saved to {saved_path}")

In [ ]:
control_utils.plot_rollout(result_static, zT=zT, zdT=zdT,
u_init=u_init,
u_final=u_final,
)


### Multiple

In [ ]:
# --- MULTI-STEP CONTROL WITH PROPER LATENT SIZE FOR KOOPMAN MODEL ---
for j in range(0,6):
    model_name = model_names[j]  # or pick one
    print(model_name)
    dyn = dynamics_models[model_name]
    dev = next(dyn.parameters()).device
    dt = configs[model_name]["delta_t"]

    # Adapt z_dim to match dyn.latent_dim (for Koopman, typically state_dim//2)
    # Many Koopman models expect input z to be shape [batch, latent_dim] (not full state)
    if hasattr(dyn, "latent_dim"):
        z_dim = dyn.latent_dim
    elif hasattr(dyn, "state_dim"):
        z_dim = dyn.state_dim // 2  # default assumption for KoopmanModel
    else:
        raise AttributeError("Cannot determine latent/state dimension for dynamics model.")

    u_dim = dyn.actuation_dim if hasattr(dyn, "actuation_dim") else 4  # fallback

    if "1 Seg" in model_name:
        T = 100
        n_iter = 300
        lr = 1e-1
        scheduler_step_size = 300
        scheduler_gamma = 0.1

        w_Q=1.0
        w_dQ=.1*dt
        w_Qf=.0
        w_dQf=.0*dt
        w_R=1e2
        delta_u_max=1.0*0.02
        w_du_bound=1e3

        dim_u=2
        dim_z=8

    elif "2 Seg" in model_name:
        T = 100
        n_iter = 300
        lr = 1e-1
        scheduler_step_size = 500
        scheduler_gamma = 0.1

        w_Q=1.0
        w_dQ=.1*dt
        w_Qf=.0
        w_dQf=.0*dt
        w_R=1e2
        delta_u_max=1.0*0.02
        w_du_bound=1e3

        dim_u=4
        dim_z=10


    # Define the resting state and initial control
    z_rest = all_target_states['Rest_States'][model_name]['z'].squeeze(0).to(dev)
    u_rest = torch.ones(u_dim, device=dev) * 0.001

    # Get all target zs, us, os (first 8 targets)
    target_file = '1Seg_Koopman_Static_Extrapolated_1'
    # 'Static_Normal'
    # '2Seg_Koopman_Static_Extrapolated_1'

    if "Normal" in target_file:
        type = 'normal'
    elif "Extrapolated" in target_file:
        type = 'extrapolated'

    if type == 'normal':
        all_zs_full = all_target_states[target_file]['z'][model_name][:8]  # might be [n, state_dim]
        all_us = all_target_states[target_file]['u'][model_name][:8]
        all_os = all_target_states[target_file]['o'][model_name]
        num_targets = 8


    elif type == 'extrapolated':
        all_zs_full=all_target_states[target_file][model_name]['z']
        all_us=None
        u_final=None
        num_targets = 5

    # Koopman expects first half of z to be 'position' (latent), second half is velocity
    # We always use only the first z_dim dims for transitions (for z, zT, etc)
    def to_latent(z):
        if z.shape[-1] > z_dim:
            return z[..., :z_dim]
        return z

    all_zs = [to_latent(z) for z in all_zs_full]

    multi_step_results = []

    z_curr = z_rest.clone()
    u_curr = u_rest.clone()
    zd_curr = torch.zeros_like(z_curr, device=dev)   # always 0

    for step in range(num_targets):
        zT_full = all_zs[step].to(dev)
        zT = to_latent(zT_full)
        zdT = torch.zeros_like(zT, device=dev)

        if type == 'normal':
            uT = all_us[step].to(dev)
            oT = all_os[step].to(dev)
        elif type == 'extrapolated':
            uT = None
            oT = None

        # Step 0 should match single static run (same z0=z_rest, zT=all_zs[0], u_init=u_rest). Sanity check:
        if step == 0:
            assert z_curr.shape == z_rest.shape and torch.allclose(z_curr, z_rest), "step 0 z0 should equal z_rest"
            assert u_curr.shape == u_rest.shape and torch.allclose(u_curr, u_rest), "step 0 u_init should equal u_rest"
            zT_static = to_latent(all_zs_full[0].to(dev))
            assert zT.shape == zT_static.shape and torch.allclose(zT, zT_static), "step 0 zT should match static zT (to_latent(all_zs[0]))"

        result = control_utils.optimize_open_loop_control(
            dyn=dyn,
            z0=z_curr,
            zd0=zd_curr,
            zT=zT,
            zdT=zdT,
            u_init=u_curr,
            T=T,
            dt=dt,
            n_iter=n_iter,
            lr=lr,
            scheduler_step_size=scheduler_step_size,
            scheduler_gamma=scheduler_gamma,
            fix_u0_to_uref=True,
            use_sigmoid=True,
            z_ref_traj=None,
            zd_ref_traj=None,
            w_Q=w_Q,
            w_dQ=w_dQ,
            w_Qf=w_Qf,
            w_dQf=w_dQf,
            w_R=w_R,
            delta_u_max=delta_u_max,
            w_du_bound=w_du_bound,
            device=dev,
            z_stats=z_stats.get(model_name) if "z_stats" in dir() else None,
            u_max=0.9,
        )
        multi_step_results.append(result)
        # Prepare next step initial state: terminal predicted latent and last control
        # result['zT_pred'] is the final state only, shape [latent_dim]
        z_curr = result['zT_pred'].detach()
        u_curr = result['u_seq'][-1].detach()
        zd_curr = torch.zeros_like(z_curr, device=dev)  # always reset to 0

        print(f"Step {step+1}/{num_targets}: Terminal error = {torch.norm(z_curr - zT).item():.4f}")

    # Go back to rest:
    zT = z_rest
    zdT = torch.zeros_like(zT, device=dev)
    uT = u_rest

    result = control_utils.optimize_open_loop_control(
        dyn=dyn,
        z0=z_curr,
        zd0=zd_curr,
        zT=zT,
        zdT=zdT,
        u_init=u_curr,
        T=T,
        dt=dt,
        n_iter=n_iter,
        lr=lr,
        scheduler_step_size=scheduler_step_size,
        scheduler_gamma=scheduler_gamma,
        fix_u0_to_uref=True,
        use_sigmoid=True,
        z_ref_traj=None,
        zd_ref_traj=None,
        w_Q=w_Q,
        w_dQ=w_dQ,
        w_Qf=w_Qf,
        w_dQf=w_dQf,
        w_R=w_R,
        delta_u_max=delta_u_max,
        w_du_bound=w_du_bound,
        device=dev,
        z_stats=z_stats.get(model_name) if "z_stats" in dir() else None,
        u_max=0.9,
    )
    multi_step_results.append(result)
    z_curr = result['zT_pred'].detach()  # final state only (1D), do not use [-1]
    print(f"Final step (rest): Terminal error = {torch.norm(z_curr - zT).item():.4f}")

    for i in range(len(multi_step_results)):
        saved_path = control_utils.save_control_output(
            model_name, target_file, multi_step_results[i],
            name_suffix=f"_{i}",
            simulator_pickle_paths=simulator_pickle_paths_2seg,
        )
        print(f"Saved to {saved_path}")


In [ ]:
# -- Concatenate all u_seq from multi_step_results (and final result), with each "step" section repeated 100 times the final control is held constant for the in-between states (not after final)
# Also construct u_real_cat_long containing the true "real" u at the end of each step for equivalent plotting

n_idle = 200

# Gather all u_seq and prepare concatenated lists
u_seqs_cat = []
u_real_cat = []
has_real = ('all_us' in locals() or 'all_us' in globals()) and all_us is not None

for idx, result in enumerate(multi_step_results):
    u_seq = result['u_seq']  # [T, act_dim]
    # For all but last multi_step result, repeat the last control 100 times after u_seq, and store only first T-1 directly
    if idx < len(multi_step_results) - 1:
        # u_seq[:-1] is the sequence excluding final action of this stage (which will be held as constant)
        u_seqs_cat.append(u_seq[:-1])
        # Hold last control for 100 steps as "dwell"
        last_control = u_seq[-1].unsqueeze(0)  # [1, act_dim]
        u_seqs_cat.append(last_control.repeat(n_idle, 1))
        if has_real:
            # Also save the corresponding real u for the final target of this episode
            # all_us has the real u for each target (sequence matches episode order)
            u_real_episode = all_us[idx].unsqueeze(0).to(u_seq.device)  # [1, act_dim]
            # Insert dwell for comparison (match the dwell timing for plotting lines)
            u_real_cat.append(u_real_episode.repeat(u_seq[:-1].shape[0], 1))
            u_real_cat.append(u_real_episode.repeat(n_idle, 1))
    else:
        # For final step, append the entire u_seq (no dwell after "rest")
        u_seqs_cat.append(u_seq)
        if has_real:
            u_real_final = u_rest.unsqueeze(0).to(u_seq.device)  # For rest, expected is u_rest
            u_real_cat.append(u_real_final.repeat(u_seq.shape[0], 1))

# Concatenate along time axis
u_seqs_cat_long = torch.cat(u_seqs_cat, dim=0)
if has_real:
    u_real_cat_long = torch.cat(u_real_cat, dim=0)
else:
    u_real_cat_long = None

# Save the concatenated control outputs as NPZ with model name and "Static_Normal" in file name
import os

# output_dir = "results/control_outputs"
# os.makedirs(output_dir, exist_ok=True)
# npz_path = os.path.join(
#     output_dir,
#     f"{model_name}_Static_Normal_controls.npz"
# )

# if u_real_cat_long is not None:
#     np.savez(
#         npz_path,
#         u_seqs_cat_long=u_seqs_cat_long.detach().cpu().numpy(),
#         u_real_cat_long=u_real_cat_long.detach().cpu().numpy(),
#         model_name=model_name
#     )
# else:
#     np.savez(
#         npz_path,
#         u_seqs_cat_long=u_seqs_cat_long.detach().cpu().numpy(),
#         model_name=model_name
#     )

#print(f"Saved control outputs to {npz_path}")
print(f"Concatenated control sequence shape: {u_seqs_cat_long.shape}")
if u_real_cat_long is not None:
    print(f"Concatenated real control (target) sequence shape: {u_real_cat_long.shape}")
else:
    print("No real control (target) available.")

In [ ]:
num_ctrl = u_seqs_cat_long.shape[1]
colors = plt.cm.tab10.colors if num_ctrl <= 10 else plt.cm.nipy_spectral(np.linspace(0, 1, num_ctrl))  # pick a colormap

for i in range(num_ctrl):
    c = colors[i % len(colors)]
    plt.plot(u_seqs_cat_long[:, i].detach().cpu().numpy(), color=c, label=f"Pred Control {i}")
    if u_real_cat_long is not None:
        plt.plot(u_real_cat_long[:, i].detach().cpu().numpy(), linestyle='--', color=c, label=f"Real Control {i} (target)")


In [ ]:
# Plot decoded images versus real target observations at the end of each multi-step sub-trajectory

vae = vaes[model_name]
vae.eval()

# Gather all real target oT images and predicted z_traj[-1] decoded images for each step
real_targets = []
decoded_imgs = []

with torch.no_grad():
    for step, result in enumerate(multi_step_results):
        # For all steps except the last: real target from all_os
        if step < len(multi_step_results) - 1:
            oT = all_os[step]
            real_targets.append(oT.cpu())
        else:
            # For the last step, use decoded image from rest state (same as simulator)
            z_rest = all_target_states['Rest_States'][model_name]['z'].to(next(vae.parameters()).device)
            rest_img = vae.decode(z_rest).detach().cpu().squeeze()
            real_targets.append(rest_img)
        # decode predicted z_traj[-1] from this sub-trajectory
        z_traj = result['z_traj']
        decoded_img = vae.decode(z_traj[-1].unsqueeze(0)).detach().cpu().squeeze()
        decoded_imgs.append(decoded_img)

# Determine figure shape: show all steps (num_targets) in a grid (real, decoded)
nsteps = len(real_targets)
fig, axes = plt.subplots(nsteps, 2, figsize=(5, 2.5 * nsteps))
if nsteps == 1:
    axes = axes.reshape(1, 2)

for step in range(nsteps):
    # Left: real target
    ax0 = axes[step, 0]
    img = real_targets[step]
    if isinstance(img, torch.Tensor):
        img = img.squeeze()
        if img.dim() == 3:
            img = img.permute(1, 2, 0)
        npimg = img.numpy()
    else:
        npimg = img
    npimg = npimg.clip(0, 1)
    if step == nsteps - 1:
        ax0.set_title(f"Step {step+1}: Rest (decoded rest state)")
    else:
        ax0.set_title(f"Step {step+1}: Real oT")
    ax0.imshow(npimg, cmap=None if npimg.ndim == 3 else "gray")
    ax0.axis('off')

    # Right: decoded
    ax1 = axes[step, 1]
    img = decoded_imgs[step]
    if isinstance(img, torch.Tensor):
        img = img.squeeze()
        if img.dim() == 3:
            img = img.permute(1, 2, 0)
        npimg = img.numpy()
    else:
        npimg = img
    npimg = npimg.clip(0, 1)
    ax1.imshow(npimg, cmap=None if npimg.ndim == 3 else "gray")
    ax1.set_title(f"Step {step+1}: Decoded z_traj[-1]")
    ax1.axis('off')

plt.tight_layout()
plt.show()


In [ ]:
# Print image RMSE between real oT and predicted oT (decoded from last z_traj), then plot 2x2 images
vae = vaes[model_name]
vae.eval()
with torch.no_grad():
    z_traj = result_static['z_traj']
    z_traj_0_dec = vae.decode(z_traj[0].unsqueeze(0)).detach().cpu()
    z_traj_T_dec = vae.decode(z_traj[-1].unsqueeze(0)).detach().cpu()

# Compute image RMSE
rmse = torch.sqrt(torch.mean((oT.cpu().float().view(-1) - z_traj_T_dec.cpu().float().view(-1)) ** 2)).item()
print(f"Image RMSE between real oT and predicted oT: {rmse:.5f}")

# Plot
import matplotlib.pyplot as plt
fig, axs = plt.subplots(2, 2, figsize=(5, 5))
titles = [
    "o0 (reference observation)",
    "oT (reference observation)",
    "z_traj[0] decoded\n(rollout)",
    "z_traj[-1] decoded\n(rollout)"
]
ims = [o0.detach().cpu(), oT.detach().cpu(), z_traj_0_dec.detach().cpu(), z_traj_T_dec.detach().cpu()]

for i, (im, title) in enumerate(zip(ims, titles)):
    ax = axs[i // 2, i % 2]
    img = im
    if isinstance(img, torch.Tensor):
        img = img.squeeze()
        if img.dim() == 3:
            img = img.permute(1, 2, 0)
        npimg = img.numpy()
    elif isinstance(img, np.ndarray):
        npimg = img.squeeze()
        if npimg.ndim == 3 and npimg.shape[0] in [1, 3]:
            if npimg.shape[0] == 1:
                npimg = npimg[0]
            elif npimg.shape[0] == 3:
                npimg = np.transpose(npimg, (1, 2, 0))
    else:
        raise ValueError(f"Image type not supported: {type(img)}")
    npimg = npimg.clip(0, 1)
    ax.imshow(npimg, cmap=None if npimg.ndim == 3 else "gray")
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.show()


In [ ]:
print(u_seq_static[-1])
print(u_final)
# Print error between both u's
u1 = u_seq_static[-1]
u2 = u_final
u_error = np.mean((u1 - u2).detach().cpu().numpy()**2)
print(u_error)

In [ ]:
# Render GIF of static tracking trajectory with green frame numbers
try:
    import imageio
except ImportError:
    print("imageio not found. Installing...")
    import subprocess
    subprocess.check_call(["pip", "install", "imageio"])
    import imageio

vae = vaes[model_name]
vae.eval()

# Also need PIL for text annotation
try:
    from PIL import Image, ImageDraw, ImageFont
except ImportError:
    print("PIL not found. Installing...")
    import subprocess
    subprocess.check_call(["pip", "install", "pillow"])
    from PIL import Image, ImageDraw, ImageFont

# Determine which control sequence to use
use_linear_control = configs[model_name].get("use_linear_control", False)
u_seq_use = None

if use_linear_control:
    # Linear control: use u_seq from linear optimization cell
    if 'u_seq' in locals():
        u_seq_use = u_seq.detach().to(dev)
    else:
        print("Linear control: u_seq not found. Run the linear optimization cell first.")
else:
    u_seq_use = u_seq_static.detach().to(dev)


if u_seq_use is not None:
    # Rollout dynamics to get full trajectory
    z_curr = z0.to(dev)
    zd_curr = zd0.to(dev)
    
    z_traj_list = [z_curr]  # Start with initial state
    
    with torch.no_grad():
        for t in range(T):
            u_t = u_seq_use[t:t+1]
            z_next, zd_next = dyn.forward(z_curr.unsqueeze(0), zd_curr.unsqueeze(0), u_t, dt)
            z_curr = z_next.squeeze(0)
            zd_curr = zd_next.squeeze(0)
            z_traj_list.append(z_curr)
    
    # Decode all states
    z_traj_tensor = torch.stack(z_traj_list).to(dev)
    
    with torch.no_grad():
        o_decoded = vae.decode(z_traj_tensor).detach().cpu()
    
    # Convert to numpy and normalize
    imgs = o_decoded
    if imgs.shape[1] == 1:
        imgs = imgs.repeat(1, 3, 1, 1)
    imgs = imgs.permute(0, 2, 3, 1).numpy()
    imgs = np.clip(imgs, 0, 1)
    
    # Convert to uint8 for GIF
    imgs_uint8 = (imgs * 255).astype(np.uint8)
    
    # Annotate each frame with green frame number (top-left)
    annotated_frames = []
    for frame_idx, arr in enumerate(imgs_uint8):
        img_pil = Image.fromarray(arr)
        draw = ImageDraw.Draw(img_pil)
        # Try to get a reasonable font; fallback to default if not
        try:
            # This path will work on many systems; customize if needed
            font = ImageFont.truetype("arial.ttf", size=10)
        except Exception:
            font = ImageFont.load_default()
        text = str(frame_idx)
        # Choose green, e.g., RGB (0, 255, 0)
        draw.text((1, 1), text, font=font, fill=(0, 255, 0))
        annotated_frames.append(np.array(img_pil))
    
    # Create GIF from annotated frames
    gif_path = f"trajectory_{model_name.replace(' ', '_')}_static_tracking.gif"
    imageio.mimsave(gif_path, annotated_frames, duration=0.1, loop=0)
    
    print(f"GIF saved to: {gif_path}")
    print(f"Total frames: {len(annotated_frames)}")
    print(f"Trajectory type: static tracking")
else:
    print("Please run the optimization cell first to generate the control sequence.")

In [ ]:
# Roll out latent trajectory with constant u_init to check equilibrium
# Compatible with both linear and MLP control

num_delays = configs[model_name].get("num_actuation_delays", 1)
use_linear_control = configs[model_name].get("use_linear_control", False)

# Use constant u_init for all steps to check equilibrium
u_use = u_init.repeat(T, 1).detach().to(dev)

delta_t = configs[model_name]["delta_t"]

# Use already extracted z0, zd0, zT, zdT from existing context
z_traj = [z0]
z_dot_traj = [zd0]
with torch.no_grad():
    z_t = z0
    z_dot_t = zd0
    for t in range(T):
        u_t = u_use[t:t+1]  # Constant u_init for equilibrium check
        z_next, z_dot_next = dyn(z_t.unsqueeze(0), z_dot_t.unsqueeze(0), u_t, dt=delta_t)
        z_t = z_next.squeeze(0)
        z_dot_t = z_dot_next.squeeze(0)
        z_traj.append(z_t)
        z_dot_traj.append(z_dot_t)

z_traj = torch.stack(z_traj).detach().cpu()
z_dot_traj = torch.stack(z_dot_traj).detach().cpu()
u_pred = u_use.detach().cpu()

# Real initial and final latent states from dataset
z_real_init = z0.detach().cpu()
z_real_final = zT.detach().cpu()
z_dot_real_init = zd0.detach().cpu()
latent_dim = z_real_init.shape[0]
z_dot_real_final = zdT.detach().cpu()

# Real initial and final controls
u_real_init = u_init.detach().cpu()
u_real_final = u_final.detach().cpu()

# Handle delayed actuation if needed
base_act_dim = configs[model_name]["actuation_dim"]
if num_delays > 1:
    u_pred_blocks = u_pred.view(T, num_delays, base_act_dim)
    u_pred_plot = u_pred_blocks[:, -1, :]  # current u_t
else:
    u_pred_plot = u_pred

fig, axes = plt.subplots(3, 1, figsize=(6, 6), sharex=True)

# Latent z as line plots
colors_z = plt.rcParams['axes.prop_cycle'].by_key()['color']
for d in range(z_traj.shape[1]):
    c = colors_z[d % len(colors_z)]
    axes[0].plot(z_traj[:, d], color=c, alpha=0.8)
    axes[0].plot(0, z_real_init[d].item(), marker='o', color=c, markerfacecolor='none', markersize=8, label=f"z_real init {d}" if d==0 else None)
    # Add horizontal dashed line at initial value to check if it stays constant
    axes[0].axhline(
        y=z_real_init[d].item(),
        color=c,
        linestyle=':',
        linewidth=1,
        alpha=0.7,
        label=None
    )

axes[0].set_title("Latent z: with constant u_init (should stay at initial if equilibrium)")
axes[0].set_ylabel("value")
axes[0].legend()

# Latent z_dot as line plots
for d in range(z_dot_traj.shape[1]):
    c = colors_z[d % len(colors_z)]
    axes[1].plot(z_dot_traj[:, d], color=c, alpha=0.8)
    axes[1].plot(0, z_dot_real_init[d].item(), marker='o', color=c, markerfacecolor='none', markersize=8, label=f"z_dot_real init {d}" if d==0 else None)
    # Add horizontal dashed line at zero (should stay at zero for equilibrium)
    axes[1].axhline(
        y=0.0,
        color=c,
        linestyle=':',
        linewidth=1,
        alpha=0.7,
        label=None
    )

axes[1].set_title("Latent z_dot: with constant u_init (should stay at zero if equilibrium)")
axes[1].set_ylabel("value")
axes[1].legend()

# Control u plots (should be constant)
t = np.arange(T)
colors_u = colors_z
for i in range(u_pred_plot.shape[1]):
    c = colors_u[i % len(colors_u)]
    axes[2].plot(t, u_pred_plot[:, i], color=c, alpha=0.8, label=f"u_{i}")
    axes[2].plot(0, u_real_init[i].item(), marker='o', color=c, markerfacecolor='none', markersize=8)
    # Add horizontal dashed line at initial value
    axes[2].axhline(
        y=u_real_init[i].item(),
        color=c,
        linestyle=':',
        linewidth=1,
        alpha=0.7,
        label=None
    )

axes[2].set_title("Control u: constant u_init (equilibrium check)")
axes[2].set_xlabel("step")
axes[2].set_ylabel("u")
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Render GIF of constant u_init equilibrium test with green frame numbers
try:
    import imageio
except ImportError:
    print("imageio not found. Installing...")
    import subprocess
    subprocess.check_call(["pip", "install", "imageio"])
    import imageio

vae = vaes[model_name]
vae.eval()

# Also need PIL for text annotation
try:
    from PIL import Image, ImageDraw, ImageFont
except ImportError:
    print("PIL not found. Installing...")
    import subprocess
    subprocess.check_call(["pip", "install", "pillow"])
    from PIL import Image, ImageDraw, ImageFont

# Use constant u_init for all steps to check equilibrium
u_use = u_init.repeat(T, 1).detach().to(dev)

delta_t = configs[model_name]["delta_t"]

# Rollout dynamics with constant u_init
z_curr = z0.to(dev)
zd_curr = zd0.to(dev)

z_traj_list = [z_curr]  # Start with initial state

with torch.no_grad():
    for t in range(T):
        u_t = u_use[t:t+1]  # Constant u_init for equilibrium check
        z_next, zd_next = dyn.forward(z_curr.unsqueeze(0), zd_curr.unsqueeze(0), u_t, dt)
        z_curr = z_next.squeeze(0)
        zd_curr = zd_next.squeeze(0)
        z_traj_list.append(z_curr)

# Decode all states
z_traj_tensor = torch.stack(z_traj_list).to(dev)

with torch.no_grad():
    o_decoded = vae.decode(z_traj_tensor).detach().cpu()

# Convert to numpy and normalize
imgs = o_decoded
if imgs.shape[1] == 1:
    imgs = imgs.repeat(1, 3, 1, 1)
imgs = imgs.permute(0, 2, 3, 1).numpy()
imgs = np.clip(imgs, 0, 1)

# Convert to uint8 for GIF
imgs_uint8 = (imgs * 255).astype(np.uint8)

# Annotate each frame with green frame number (top-left)
annotated_frames = []
for frame_idx, arr in enumerate(imgs_uint8):
    img_pil = Image.fromarray(arr)
    draw = ImageDraw.Draw(img_pil)
    # Try to get a reasonable font; fallback to default if not
    try:
        # This path will work on many systems; customize if needed
        font = ImageFont.truetype("arial.ttf", size=10)
    except Exception:
        font = ImageFont.load_default()
    text = str(frame_idx)
    # Choose green, e.g., RGB (0, 255, 0)
    draw.text((1, 1), text, font=font, fill=(0, 255, 0))
    annotated_frames.append(np.array(img_pil))

# Create GIF from annotated frames
gif_path = f"trajectory_{model_name.replace(' ', '_')}_constant_u.gif"
imageio.mimsave(gif_path, annotated_frames, duration=0.1, loop=0)

print(f"GIF saved to: {gif_path}")
print(f"Total frames: {len(annotated_frames)}")
print(f"Trajectory type: constant u_init (equilibrium test)")

## Dynamic: Normal

In [ ]:
for j in range(0,6):

    model_name = model_names[j]  # or pick one
    print(model_name)
    dyn = dynamics_models[model_name]
    dev = next(dyn.parameters()).device
    dt = configs[model_name]["delta_t"]


    if "1 Seg" in model_name:
        T = 200
        n_iter = 300
        lr = 1e-1
        scheduler_step_size = 500
        scheduler_gamma = 0.1

        w_Q=1.0
        w_dQ=.0*dt
        w_dQl=0.1*dt
        w_Qf=.0
        w_dQf=.0*dt
        w_R=1e2
        delta_u_max=1.0*0.02
        w_du_bound=1e3

        dim_u=2
        dim_z=8

    elif "2 Seg" in model_name:
        T = 200
        n_iter = 300
        lr = 1e-1
        scheduler_step_size = 500
        scheduler_gamma = 0.1

        w_Q=1.0
        w_dQ=.0*dt
        w_dQl=0.1*dt
        w_Qf=.0
        w_dQf=.0*dt
        w_R=1e2
        delta_u_max=1.0*0.02
        w_du_bound=1e3

        dim_u=4
        dim_z=10


    z_rest = all_target_states['Rest_States'][model_name]['z'].squeeze(0).to(dev)
    u_rest=torch.ones(dim_u,device=dev)*0.001

    current_target_state=0
    for i in range(1,6):
        target_file=f'1Seg_Koopman_Dynamic_Normal_{i}'

        z_dyn_target=all_target_states[target_file][model_name]['z']
        zd_dyn_target=all_target_states[target_file][model_name]['z_dot']

        # Use same z_dim/to_latent as multi-step so step-0 target matches exactly
        def to_latent_static(z):
            z = z.to(dev) if hasattr(z, 'to') else torch.tensor(z, device=dev)
            if z.shape[-1] > dim_z:
                return z[..., :dim_z].clone()
            return z

        z0=z_rest
        u_init=u_rest


        zd0 = torch.zeros(dim_z,device=dev) # Should always start at 0 speed

        zT = z_dyn_target.to(dev)
        zdT = zd_dyn_target.to(dev)

        z_traj = torch.cat([z0.unsqueeze(0), zT, z0.unsqueeze(0)], dim=0)
        zd_traj = torch.cat([z0.unsqueeze(0), zdT, z0.unsqueeze(0)], dim=0)

        result_dynamic = control_utils.optimize_open_loop_control(
            dyn=dyn,
            z0=z0,
            zd0=zd0,
            zT=z0,
            zdT=zd0,
            u_init=u_init,
            T=T,
            dt=dt,
            n_iter=n_iter,
            lr=lr,
            scheduler_step_size=scheduler_step_size,
            scheduler_gamma=scheduler_gamma,
            fix_u0_to_uref=True,
            use_sigmoid=True,
            z_ref_traj=z_traj,
            zd_ref_traj=zd_traj,
            w_Q=w_Q,
            w_dQ=w_dQ,
            w_dQl=w_dQl,
            w_Qf=w_Qf,
            w_dQf=w_dQf,
            w_R=w_R,
            device=dev,
            z_stats=z_stats.get(model_name) if "z_stats" in dir() else None,
            delta_u_max=delta_u_max,
            w_du_bound=w_du_bound,
            u_max=0.9,
            waypoint_window="symmetric",
        )

        u_seq_dynamic = result_dynamic["u_seq"]
        print(f"Dynamic control: final terminal error = {torch.norm(result_dynamic['zT_pred'] - zT).item():.4f}")

        saved_path = control_utils.save_control_output(
            model_name, target_file, result_dynamic,
            simulator_pickle_paths=simulator_pickle_paths_2seg,
        )
        print(f"Saved to {saved_path}")


In [ ]:
import matplotlib.pyplot as plt

obs = all_target_states[target_file][model_name]['obs']  # Tensor [N, 1, H, W] or [N, H, W]
if isinstance(obs, torch.Tensor):
    obs = obs.cpu().numpy()
if obs.ndim == 4:
    obs = obs[:, 0]  # Remove channel dim if present

N = obs.shape[0]
fig, axs = plt.subplots(1, 8, figsize=(16, 2))
for i in range(8):
    ax = axs[i]
    if i < N:
        ax.imshow(obs[i], cmap='gray')
    ax.axis('off')
    ax.set_title(f'Obs {i}')
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

vae = vaes[model_name]
z_traj_full = result_dynamic['z_traj']  # Tensor [T, latent_dim]

T = z_traj_full.shape[0]
num_imgs = 10
indices = np.linspace(0, T-1, num_imgs, dtype=int)

fig, axs = plt.subplots(1, num_imgs, figsize=(2*num_imgs, 2))
for i, idx in enumerate(indices):
    z = z_traj_full[idx]
    with torch.no_grad():
        img = vae.decode(z.unsqueeze(0)).squeeze(0).detach().cpu()
    # handle channel logic (C,H,W) or (H,W)
    if img.ndim == 3 and img.shape[0] in [1, 3]:
        # [C, H, W], want to show grayscale if C==1, else as RGB
        if img.shape[0] == 1:
            show_img = img[0]
            cmap = 'gray'
        else:
            show_img = img.permute(1,2,0).numpy()
            cmap = None
    else:
        show_img = img.numpy()
        cmap = 'gray' if show_img.ndim == 2 else None
        
    ax = axs[i]
    if cmap:
        ax.imshow(show_img, cmap=cmap)
    else:
        ax.imshow(show_img)
    ax.axis('off')
    ax.set_title(f't={idx}')
plt.tight_layout()
plt.show()

In [ ]:
control_utils.plot_rollout(result_dynamic, zT=None, zdT=None,z_ref_traj=z_traj,zd_ref_traj=zd_traj,
u_init=u_init,
)


## Dynamic: High dynamics

In [ ]:
for j in range(0,6):
    model_name = model_names[j]  # or pick one
    print(model_name)
    dyn = dynamics_models[model_name]
    dev = next(dyn.parameters()).device
    dt = configs[model_name]["delta_t"]


    if "1 Seg" in model_name:
        T = 150
        n_iter = 300
        lr = 1e-1
        scheduler_step_size = 500
        scheduler_gamma = 0.1

        w_Q=1.0
        w_dQ=.0*dt
        w_dQl=0.1*dt
        w_Qf=.0
        w_Qfl=0.0
        w_dQf=.0*dt
        w_R=1e2
        delta_u_max=1.5*0.02
        w_du_bound=1e3

        dim_u=2
        dim_z=8

    elif "2 Seg" in model_name:
        T = 150
        n_iter = 300
        lr = 1e-1
        scheduler_step_size = 500
        scheduler_gamma = 0.1

        w_Q=1.0
        w_dQ=.0*dt
        w_dQl=0.1*dt
        w_Qf=.0
        w_Qfl=0.0
        w_dQf=.0*dt
        w_R=1e2
        delta_u_max=1.5*0.02
        w_du_bound=1e3

        dim_u=4
        dim_z=10


    z_rest = all_target_states['Rest_States'][model_name]['z'].squeeze(0).to(dev)
    u_rest=torch.ones(dim_u,device=dev)*0.001

    current_target_state=0

    for i in range(1,4):

        target_file=fr'1Seg_Koopman_Dynamic_HighDyn_{i}'
        z_dyn_target=all_target_states[target_file][model_name]['z']
        zd_dyn_target=all_target_states[target_file][model_name]['z_dot']

        # Use same z_dim/to_latent as multi-step so step-0 target matches exactly
        def to_latent_static(z):
            z = z.to(dev) if hasattr(z, 'to') else torch.tensor(z, device=dev)
            if z.shape[-1] > dim_z:
                return z[..., :dim_z].clone()
            return z

        z0=z_rest
        u_init=u_rest


        zd0 = torch.zeros(dim_z,device=dev) # Should always start at 0 speed

        zT = z_dyn_target.to(dev)
        zdT = zd_dyn_target.to(dev)

        z_traj = torch.cat([z0.unsqueeze(0), zT, z0.unsqueeze(0)], dim=0)
        zd_traj = torch.cat([zd0.unsqueeze(0), zdT, zd0.unsqueeze(0)], dim=0)

        result_dynamic = control_utils.optimize_open_loop_control(
            dyn=dyn,
            z0=z0,
            zd0=zd0,
            zT=z0,
            zdT=zd0,
            u_init=u_init,
            T=T,
            dt=dt,
            n_iter=n_iter,
            lr=lr,
            scheduler_step_size=scheduler_step_size,
            scheduler_gamma=scheduler_gamma,
            fix_u0_to_uref=True,
            use_sigmoid=True,
            z_ref_traj=z_traj,
            zd_ref_traj=zd_traj,
            w_Q=w_Q,
            w_dQ=w_dQ,
            w_dQl=w_dQl,
            w_Qf=w_Qf,
            w_Qfl=w_Qfl,
            w_dQf=w_dQf,
            w_R=w_R,
            device=dev,
            z_stats=z_stats.get(model_name) if "z_stats" in dir() else None,
            delta_u_max=delta_u_max,
            w_du_bound=w_du_bound,
            u_max=0.9,
            waypoint_window="symmetric",
        )

        u_seq_dynamic = result_dynamic["u_seq"]
        print(f"Dynamic control: final terminal error = {torch.norm(result_dynamic['zT_pred'] - zT).item():.4f}")

        saved_path = control_utils.save_control_output(
            model_name, target_file, result_dynamic,
            simulator_pickle_paths=simulator_pickle_paths_2seg,
        )
        print(f"Saved to {saved_path}")


In [ ]:
import matplotlib.pyplot as plt

obs = all_target_states[target_file][model_name]['obs']  # Tensor [N, 1, H, W] or [N, H, W]
if isinstance(obs, torch.Tensor):
    obs = obs.cpu().numpy()
if obs.ndim == 4:
    obs = obs[:, 0]  # Remove channel dim if present

N = obs.shape[0]
fig, axs = plt.subplots(1, 8, figsize=(16, 2))
for i in range(8):
    ax = axs[i]
    if i < N:
        ax.imshow(obs[i], cmap='gray')
    ax.axis('off')
    ax.set_title(f'Obs {i}')
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

vae = vaes[model_name]
z_traj_full = result_dynamic['z_traj']  # Tensor [T, latent_dim]

T = z_traj_full.shape[0]
num_imgs = 10
indices = np.linspace(0, T-1, num_imgs, dtype=int)

fig, axs = plt.subplots(1, num_imgs, figsize=(2*num_imgs, 2))
for i, idx in enumerate(indices):
    z = z_traj_full[idx]
    with torch.no_grad():
        img = vae.decode(z.unsqueeze(0)).squeeze(0).detach().cpu()
    # handle channel logic (C,H,W) or (H,W)
    if img.ndim == 3 and img.shape[0] in [1, 3]:
        # [C, H, W], want to show grayscale if C==1, else as RGB
        if img.shape[0] == 1:
            show_img = img[0]
            cmap = 'gray'
        else:
            show_img = img.permute(1,2,0).numpy()
            cmap = None
    else:
        show_img = img.numpy()
        cmap = 'gray' if show_img.ndim == 2 else None
        
    ax = axs[i]
    if cmap:
        ax.imshow(show_img, cmap=cmap)
    else:
        ax.imshow(show_img)
    ax.axis('off')
    ax.set_title(f't={idx}')
plt.tight_layout()
plt.show()

In [ ]:
control_utils.plot_rollout(result_dynamic, zT=None, zdT=None,z_ref_traj=z_traj,zd_ref_traj=zd_traj,
u_init=u_init,
)


## Dynamic Upswing

In [ ]:
for j in range(0,6):

    model_name = model_names[j]  # or pick one
    print(model_name)
    dyn = dynamics_models[model_name]
    dev = next(dyn.parameters()).device
    dt = configs[model_name]["delta_t"]


    if "1 Seg" in model_name:
        T = 150
        n_iter = 500
        lr = 1e-1
        scheduler_step_size = 400
        scheduler_gamma = 0.1

        w_Q=.0
        w_dQ=.0*dt
        w_Qf=1.0
        w_dQf=.1*dt
        w_R=1e3
        delta_u_max=1.0*0.02
        w_du_bound=1e4

        dim_u=2
        dim_z=8

    elif "2 Seg" in model_name:
        T = 150
        n_iter = 500
        lr = 1e-1
        scheduler_step_size = 400
        scheduler_gamma = 0.1

        w_Q=.0
        w_dQ=.0*dt
        w_Qf=1.0
        w_dQf=.1*dt
        w_R=1e3
        delta_u_max=1.0*0.02
        w_du_bound=1e4

        dim_u=4
        dim_z=10


    z_rest = all_target_states['Rest_States'][model_name]['z'].squeeze(0).to(dev)
    u_rest=torch.ones(dim_u,device=dev)*0.001

    current_target_state=0

    for i in range(1,4):

        target_file=fr'1Seg_Koopman_Dynamic_Upswing_{i}'

        upswing_z=all_target_states[target_file][model_name]['z']
        upswing_zd=all_target_states[target_file][model_name]['z_dot']

        # Use same z_dim/to_latent as multi-step so step-0 target matches exactly
        def to_latent_static(z):
            z = z.to(dev) if hasattr(z, 'to') else torch.tensor(z, device=dev)
            if z.shape[-1] > dim_z:
                return z[..., :dim_z].clone()
            return z

        z0=z_rest
        u_init=u_rest


        zd0 = torch.zeros(dim_z,device=dev) # Should always start at 0 speed

        oT = all_target_states[target_file][model_name]['obs']
        zT = upswing_z.to(dev).flatten()
        zdT = upswing_zd.to(dev).flatten()

        z_traj = torch.stack([z0, zT, z0], dim=0)
        zd_traj = torch.stack([zd0, zdT, zd0], dim=0)

        result_dynamic = control_utils.optimize_open_loop_control(
            dyn=dyn,
            z0=z0,
            zd0=zd0,
            zT=z0,
            zdT=zd0,
            u_init=u_init,
            T=T,
            dt=dt,
            n_iter=n_iter,
            lr=lr,
            scheduler_step_size=scheduler_step_size,
            scheduler_gamma=scheduler_gamma,
            fix_u0_to_uref=True,
            use_sigmoid=True,
            z_ref_traj=z_traj,
            zd_ref_traj=zd_traj,
            w_Q=w_Q,
            w_dQ=w_dQ,
            w_Qf=w_Qf,
            w_dQf=w_dQf,
            w_R=w_R,
            device=dev,
            z_stats=z_stats.get(model_name) if "z_stats" in dir() else None,
            delta_u_max=delta_u_max,
            w_du_bound=w_du_bound,
            u_max=0.9,
            waypoint_window="symmetric",
        )

        u_seq_dynamic = result_dynamic["u_seq"]
        print(f"Dynamic control: final terminal error = {torch.norm(result_dynamic['zT_pred'] - zT).item():.4f}")

        saved_path = control_utils.save_control_output(
            model_name, target_file, result_dynamic,
            simulator_pickle_paths=simulator_pickle_paths_2seg,
        )
        print(f"Saved to {saved_path}")


In [ ]:
# Plot the observation and the decoded observation from the z value at step 50

import matplotlib.pyplot as plt

# Get original observation at the target state 
obs = oT[0]
# Get the corresponding z value from the result trajectory at step 50
z_step = result_dynamic['z_traj'][T//2].unsqueeze(0)  # shape [1, latent_dim]
obs_decoded = vaes[model_name].decode(z_step).squeeze(0).detach().cpu()  # shape [C,H,W] or [H,W] depending

# Plot both images side by side
fig, axs = plt.subplots(1, 2, figsize=(8,4))
if obs.ndim == 3 and obs.shape[0] in [1,3]:  # [C,H,W]
    axs[0].imshow(obs.permute(1,2,0).numpy(), cmap='gray' if obs.shape[0] == 1 else None)
    axs[0].set_title("Original Observation")
    axs[1].imshow(obs_decoded.permute(1,2,0).numpy(), cmap='gray' if obs_decoded.shape[0] == 1 else None)
else:  # [H,W]
    axs[0].imshow(obs.numpy(), cmap='gray')
    axs[0].set_title("Original Observation")
    axs[1].imshow(obs_decoded.numpy(), cmap='gray')
axs[1].set_title("Decoded from z_traj[50]")
for ax in axs: ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
control_utils.plot_rollout(result_dynamic, zT=None, zdT=None,z_ref_traj=z_traj,zd_ref_traj=zd_traj,
u_init=u_init,
)


In [ ]:
# Create a GIF from the decoded trajectory
import matplotlib.pyplot as plt
import imageio
import numpy as np
import torch

decoded_imgs = []
vae = vaes[model_name]
for z in result_dynamic['z_traj']:
    with torch.no_grad():
        img = vae.decode(z.unsqueeze(0)).squeeze(0).detach().cpu()
    if img.ndim == 3 and img.shape[0] in [1, 3]:
        img_np = img.permute(1, 2, 0).numpy()
        if img.shape[0] == 1:
            img_np = img_np[..., 0]
    else:
        img_np = img.numpy()
    decoded_imgs.append((img_np * 255).astype(np.uint8) if img_np.max() <= 1.0 else img_np.astype(np.uint8))

gif_path = "decoded_trajectory.gif"
imageio.mimsave(gif_path, decoded_imgs, duration=0.02)  # ~15 FPS

print(f"GIF saved to: {gif_path}")

## Dynamic: Long

In [ ]:
for j in range(6,12):

    model_name = model_names[j]  # or pick one
    print(model_name)
    dyn = dynamics_models[model_name]
    dev = next(dyn.parameters()).device
    dt = configs[model_name]["delta_t"]


    T = 1000
    n_iter = 200
    lr = 1e-1
    scheduler_step_size = 500
    scheduler_gamma = 0.1

    w_Q=1.0
    w_dQ=.0*dt
    w_dQl=0.1*dt
    w_Qf=.0
    w_dQf=.0*dt
    w_R=1e2
    delta_u_max=2.0*0.02
    w_du_bound=1e3

    dim_u=4
    dim_z=10


    z_rest = all_target_states['Rest_States'][model_name]['z'].squeeze(0).to(dev)
    u_rest=torch.ones(dim_u,device=dev)*0.001

    current_target_state=0
    for i in range(1,6):
        target_file=f'2Seg_2DOscillator_Dynamic_Long_{i}'

        z_dyn_target=all_target_states[target_file][model_name]['z']
        zd_dyn_target=all_target_states[target_file][model_name]['z_dot']

        # Use same z_dim/to_latent as multi-step so step-0 target matches exactly
        def to_latent_static(z):
            z = z.to(dev) if hasattr(z, 'to') else torch.tensor(z, device=dev)
            if z.shape[-1] > dim_z:
                return z[..., :dim_z].clone()
            return z

        z0=z_rest
        u_init=u_rest


        zd0 = torch.zeros(dim_z,device=dev) # Should always start at 0 speed

        zT = z_dyn_target.to(dev)
        zdT = zd_dyn_target.to(dev)

        z_traj = torch.cat([z0.unsqueeze(0), zT, z0.unsqueeze(0)], dim=0)
        zd_traj = torch.cat([z0.unsqueeze(0), zdT, z0.unsqueeze(0)], dim=0)

        result_dynamic = control_utils.optimize_open_loop_control(
            dyn=dyn,
            z0=z0,
            zd0=zd0,
            zT=z0,
            zdT=zd0,
            u_init=u_init,
            T=T,
            dt=dt,
            n_iter=n_iter,
            lr=lr,
            scheduler_step_size=scheduler_step_size,
            scheduler_gamma=scheduler_gamma,
            fix_u0_to_uref=True,
            use_sigmoid=True,
            z_ref_traj=z_traj,
            zd_ref_traj=zd_traj,
            w_Q=w_Q,
            w_dQ=w_dQ,
            w_dQl=w_dQl,
            w_Qf=w_Qf,
            w_dQf=w_dQf,
            w_R=w_R,
            device=dev,
            z_stats=z_stats.get(model_name) if "z_stats" in dir() else None,
            delta_u_max=delta_u_max,
            w_du_bound=w_du_bound,
            u_max=0.9,
            waypoint_window="symmetric",
        )

        u_seq_dynamic = result_dynamic["u_seq"]
        print(f"Dynamic control: final terminal error = {torch.norm(result_dynamic['zT_pred'] - zT).item():.4f}")

        saved_path = control_utils.save_control_output(
            model_name, target_file, result_dynamic,
            simulator_pickle_paths=simulator_pickle_paths_2seg,
        )
        print(f"Saved to {saved_path}")


In [ ]:
control_utils.plot_rollout(result_dynamic, zT=None, zdT=None,z_ref_traj=z_traj,zd_ref_traj=zd_traj,
u_init=u_init,
)
